# Lesson 33 · Generating a sine — building the atom of sound yourself: the sine wave x[n]=A·sin(2πfn/sr)

**Goal**: write `x[n] = A·sin(2π·f·n/sr)` yourself, translating "frequency, amplitude, sample rate" into NumPy code that actually runs; then use `np.allclose` to line it up point by point against the repo's reference.

> **How this relates to L04**: it's the same formula. L04's `sinusoid(t,A,f,phi)` takes a time array and has an adjustable phase; this lesson's `my_sine` takes **freq / duration / sample_rate**, matching the repo's test signals. The two APIs don't clash.

`aurora.audio.sine` is where the test tones for the later STFT / mel lessons come from — and `my_sine` is that same formula, written by you.

← **Previous lesson** · [L32 · NumPy for signals](L32_numpy_signals.ipynb)

> Last lesson covered **NumPy for signals**: building a 16 kHz time axis with np.arange / linspace and putting a timestamp on every sample.
> This lesson explores **generating a sine**.

## This lesson's story: the sine wave is the atom of sound

Physics has an astonishing result: **any sound can be broken down into a sum of sine waves** (that's Fourier's theorem).
Middle C on a piano (261 Hz), a guitar string, a human voice — at heart they're all sine waves of different frequencies and amplitudes, stacked on top of each other.

The formula for a sine wave:

$$x[n] = A \cdot \sin\!\left(2\pi f \cdot \frac{n}{f_s}\right)$$

How to read it: at sample `n`, the circle has already turned through `2πf·n/fₛ` radians; `sin` takes the projection of that angle, and we multiply by the amplitude `A`.

| Variable | Meaning | Typical value |
|---|---|---|
| `f` | frequency (Hz), sets how high or low the pitch is | 440 (piano A4) |
| `fₛ` | sample rate (Hz), sets the time resolution | 16000 or 44100 |
| `A` | amplitude, sets the loudness | 1.0 (normalized) |
| `n` | sample index = `np.arange(N)` | 0, 1, 2, ..., N-1 |

Your task in this lesson: implement `my_sine(freq, duration, sample_rate)` and compare it point by point with `aurora.audio.sine` (Aurora's production implementation).

## 1. Import the tools + the repo's "answer key"

This section lets you write your own version while keeping the repo's version on hand to compare against at any moment. The worst way to learn is "looking at the answer without knowing why it looks that way". So we put the reference implementation right beside you first, then take the formula apart step by step.


## Groundwork: the magic linking time, frequency and radians

Before the code, we need to understand one key conversion: **how to translate "time" into "an angle on a circle".**

### From sample rate to time interval

What does a sample rate of `sr = 8 Hz` mean? "8 points every second". Put another way, the time between two adjacent points is `1/sr = 1/8 = 0.125 seconds`.

If you're at sample `n` (`n = 0, 1, 2, ...`), the corresponding time is:
$$t[n] = n \times \frac{1}{sr} = \frac{n}{sr}$$

For example, with `sr=8`:
- `n=0` → `t=0` seconds
- `n=2` → `t=2/8=0.25` seconds
- `n=4` → `t=0.5` seconds

No magic here — it's just "the point's index" times "the time width of each point".

### From frequency to how fast the circle turns

A frequency of `f = 1 Hz` means "repeats once every second", in other words "one lap of the circle every second".

1 lap = `2π` radians (that's how the circle is defined: circumference = `2πr`, and radians = circumference/radius).

So the angle turned each second is `2π × f` radians/second.

### Turning time into the angle the circle has turned

If `t` seconds have gone by, and the circle turns `2πf` radians every second, then the total angle turned is:
$$\text{angle}[n] = (2\pi f) \times t[n] = 2\pi f \times \frac{n}{sr}$$

That's exactly where the `2π·f·n/sr` in the formula comes from. It isn't "obvious"; it's **a direct conversion from time to a position on the circle.**

### Why can sin stand for a sound wave?

A point on the circle has coordinates `(cos θ, sin θ)`. Its projection onto the vertical axis is `sin θ`. As you go round the circle, the vertical value swings back and forth between `-1` and `1` — just like the diaphragm of a dynamic microphone moving back and forth with a sound wave. That's **why we use sin to generate sound waves.**

(Using `cos` works too; it's just shifted by 90° in phase. It sounds the same, but the plots are offset.)

## Into the lab: take sound apart into arrays you can look at

Generate 8 samples with `sample_rate=8`, `duration=1.0`, `freq=2.0`, printing every intermediate quantity, so you can see directly how the values of `n`, `t`, `angle` and `wave` line up.

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np
from aurora.audio import sine as reference_sine  # the repo's implementation = your answer key
print('ready')

## Hands-on: how a sequence turns into a signal, step by step

Look at the three intermediate arrays `n`, `t` and `angle`: `angle` is `t` times `2π·freq`. With `freq=2`, `angle` goes from 0 to 4π within 1 second — two full cycles.

In [ ]:
import numpy as np

sample_rate = 8
duration = 1.0
freq = 2.0
N = int(duration * sample_rate)
n = np.arange(N)
t = n / sample_rate
angle = 2 * np.pi * freq * t
wave = np.sin(angle)

print('N =', N)
print('sample indices n =', n)
print('time axis t =', np.round(t, 3))
print('angle =', np.round(angle, 3))
print('sin(angle) =', np.round(wave, 3))


## Code experiment: loop over frequency, amplitude and phase

When `freq` doubles, the number of zero crossings in the same stretch of time doubles too; `amplitude` only changes the range of the values, not the shape of the waveform.

## Digging deeper: what does "the shape doesn't change" really mean? And how does amplitude relate to loudness?

The previous section said "`amplitude` only changes the range of the values, not the shape of the waveform". The first time you read that, it's easy to be puzzled: if you draw two sine waves with different amplitudes, one swinging up and down a lot and one swinging only a little, **your eyes clearly see something different** — so how can anyone say they have "the same shape"? Let's make the word "shape" a bit more precise.

### What "the same shape" means, precisely

Here "shape" refers **only** to two things:

1. **How fast it oscillates** (frequency) — how many swings it repeats each second
2. **When it reaches its highest/lowest points** (phase) — which samples the peaks and troughs land on

It does **not** include how big the swing is — that's decided by the amplitude alone, which is a separate matter.

An analogy: blow a photo up or shrink it down as a whole, and the picture itself (the relative positions and proportions of the features in a face) doesn't change at all; only the size of the whole photo changes. Amplitude is the "scale factor" here:

$$x_A[n] = A \cdot \sin(2\pi f \cdot n/f_s), \qquad x_1[n] = 1 \cdot \sin(2\pi f \cdot n/f_s)$$

The two are really the same function scaled by a factor: `x_A[n] = A · x_1[n]`. Compare:

- How fast they oscillate is exactly the same — they use the same `f` and the same `sr`
- When they reach their highest point is exactly the same — both waves hit their peak at the same `n`
- The only change is that the values are stretched or squashed as a whole by a factor of `A` (the "height" changed, not the "rhythm")

That's why we say they have "the same shape" — **the same period and phase, just a different size**, like two prints of the same photo at different sizes.

### So does amplitude have anything to do with loudness?

This is a very natural audio question: if amplitude is just a mathematical scale factor, does it have anything to do with how loud the ear hears it?

The answer: **yes, and it's the most direct physical relationship there is.**

Digital audio ends up being sent to a speaker, where it drives the diaphragm (cone) inside back and forth. Each value in the array is essentially "where the diaphragm should be at this instant". The larger the value, the farther the diaphragm moves, the harder it vibrates, the more force it pushes the air with — and the "louder" the sound feels to the ear.

So if you change `amplitude` from `0.5` to `1.0`, it's like making the speaker's diaphragm move twice as far — and the loudness you hear goes up noticeably too. Roughly speaking:

**amplitude value ≈ loudness** (at least without special audio processing such as compression or EQ).

This echoes the line in the variable table at the start of the lesson — "`A` | amplitude, sets the loudness": amplitude isn't just an abstract mathematical scale factor; in the physical world it corresponds to exactly "how loud this sound is". The code below lets you see this directly: the bigger the amplitude, the bigger the absolute values of `min`/`max`, and the "taller" the waveform is stretched vertically — but the positions of the zero crossings (that is, the rhythm) don't move at all.

In [ ]:
import numpy as np

sample_rate = 16
duration = 0.5
t = np.arange(int(duration * sample_rate)) / sample_rate

for freq in [1, 2, 4]:
    wave = np.sin(2 * np.pi * freq * t)
    print(f'freq={freq}Hz ->', np.round(wave, 3))

for amplitude in [0.5, 1.0, 2.0]:
    wave = amplitude * np.sin(2 * np.pi * 2 * t)
    print(f'amplitude={amplitude} -> min={wave.min():.1f}, max={wave.max():.1f}')


## Key concept: full cycles and the number of samples

Before the hand-worked example, we need to clear up a common confusion: **"how many samples does one full cycle take?"**

### Definition: how a cycle relates to the number of samples

For a sine wave of frequency `f` Hz, at a sample rate of `sr` Hz:
- **cycles per second** is `f` (e.g. `f=440` means 440 cycles every second)
- **samples per second** is `sr` (e.g. `sr=16000` means 16000 points every second)
- **samples per cycle** = `sr / f`

For example:
- `f=1 Hz, sr=8 Hz` → 8 points per cycle
- `f=440 Hz, sr=16000 Hz` → 16000/440 ≈ 36.36 points per cycle (not necessarily a whole number!)

### The "start" and "end" of a full cycle

Here's a spot where people get stuck: with `N=8` samples, indexed `n=0,1,2,...,7`:
- sample 0 has `angle=0`, `sin(0)=0` (the start of the cycle)
- sample 4 has `angle=π`, `sin(π)=0` (half a cycle, a zero crossing)
- sample 7 has `angle=7π/4 ≈ 1.75π`, `sin(7π/4) ≈ -0.707` (not back to 0 yet!)

**This is key**: the last point of the array `n=0~7` (n=7) does **not** come back to the starting point (`sin=0`) — even here, where `sr/f = 8` is a whole number. The last sample always sits one step before the start of the next cycle; a whole number of cycles only means that the *next* sample (n=8) would land back on 0. This is the natural result of **truncated sampling** — your array has a fixed length, and its last point sits partway through a cycle.

It's not a bug; it's a feature.

## 2. Work out a few samples by hand

Frequency `f=1 Hz`, sample rate `fₛ=8 Hz`, amplitude `A=1`.

**Sample 0** (`n=0`): angle = 2π·1·0/8 = 0, x[0] = sin(0) = **0**
**Sample 2** (`n=2`): angle = 2π·1·2/8 = π/2, x[2] = sin(π/2) = **1** (peak)
**Sample 4** (`n=4`): angle = 2π·1·4/8 = π, x[4] = sin(π) = **0** (zero crossing)
**Sample 6** (`n=6`): angle = 2π·1·6/8 = 3π/2, x[6] = sin(3π/2) = **-1** (trough)

The 8 points make exactly one full cycle — because `fₛ/f = 8/1 = 8` (the Nyquist theorem: one full turn needs at least 2 points).

> **🔗 Aurora connection**: `aurora.audio.sine` is exactly this formula; it's the base signal for all the synthetic-audio tests, and the source of the "single pure tone" test cases in the FFT validation suite (`tests/audio/test_transforms.py`).

## 3. ✏️ Your task: implement `my_sine`

The crux here isn't the formula itself; it's knowing, as you translate the formula into code, what role each variable plays:

- `freq`: frequency, sets how fast it oscillates
- `duration`: duration, sets how long the sequence is
- `sample_rate`: sample rate (sr), sets how many points are taken each second
- `n` or `t`: the position sequence, the index or time of each sample
- `amplitude`: amplitude, sets how far the waveform swings up and down

**Line of reasoning**:
1. First compute the total number of points: `N = int(duration * sample_rate)` (truncating, consistent with the repo's `aurora.audio.sine`); this is the length of the returned array.
2. Generate the integer index sequence: `n = np.arange(N)`, from 0 to N-1, not including the endpoint N.
3. Convert the indices into angles: `angle = 2*np.pi*freq*n/sample_rate`, where `n/sample_rate` is the time `t` of each point.
4. Return `amplitude * np.sin(angle)` — amplitude is just a scale factor and doesn't change the shape of the waveform.

**Reference input and output**: `freq=1, duration=1, sample_rate=8` → 8 points, `angle=[0, π/4, π/2, 3π/4, π, 5π/4, 3π/2, 7π/4]`, `sin≈[0, 0.707, 1, 0.707, 0, -0.707, -1, -0.707]`

> 💡 Need a hint? Once you've done the exercise, you can look at the reference implementation in the `solutions/` folder.



### Three things to pin down before you write `my_sine`

- Input: `freq` (Hz), `duration` (seconds), `sample_rate` (points/second), `amplitude` (default 1.0)
- Key steps: `N = int(duration * sample_rate)`, `n = np.arange(N)`, `angle = 2*np.pi*freq*n/sample_rate`
- Return: a float64 array of length `N`, with values in `[-amplitude, amplitude]`

In [ ]:
def my_sine(freq, duration, sample_rate, amplitude=1.0):
    # ✏️ TODO: implement the sine wave from the formula and return it
    raise NotImplementedError("TODO: implement the sine wave from the formula and return it")


## 4. Check your answer: compare point by point with the repo's implementation

What to look at here isn't just "did it raise an error"; check the match on three levels:

- **Do the lengths match**: tells you whether you generated the right number of samples
- **Does every point match**: tells you whether you wrote the formula correctly
- **Is the error tiny**: tells you whether the floating-point error in the computation is within the normal range

If it doesn't pass, don't rush to change the `sin` itself; first check that `n` starts at 0, that `N` is computed correctly, and that you didn't divide by the sample rate the wrong way round.


## Why `int()` and not `round()`?

To compute the total number of samples `N = int(duration * sample_rate)` we use **truncation** (`int()`), not **rounding to nearest** (`round()`). There's a real reason for that choice.

### How the two kinds of rounding differ

Say `duration=0.5 s, sample_rate=3 Hz`:
- `duration * sample_rate = 1.5`
- `int(1.5) = 1` (truncation)
- `round(1.5) = 2` (rounding to nearest)

Which one is right?

### Understanding it physically

`duration=0.5 s` means "we want the samples that fit in this length of time". At `sr=3 Hz`, the sample positions on the time axis are:
- sample 0: `t=0.0 s` (in range ✓)
- sample 1: `t=1/3 ≈ 0.333 s` (in range ✓)
- sample 2: `t=2/3 ≈ 0.667 s` (**out of range** ✗ — greater than 0.5)

So two sample *instants* fall inside — counting instants alone would argue for 2. The reason for `int()` is that each sample stands for a slice of time `1/sr` long, so N samples make a clip `N/sr` seconds long. With N=2 the clip would last 2/3 s, longer than the 0.5 s asked for; with N=1 it lasts 1/3 s, which fits. That gives `int(1.5)=1`.

This is called the **half-open interval** `[0, duration)`: the clip includes its start and never runs past its end.

### From an audio-processing point of view

In audio frameworks, the usual time-axis convention is "**cover exactly the time span you specified, never more**". If you say "generate 0.5 seconds of audio", you mean "give me the samples that fit inside 0.5 seconds", not "get as close to 0.5 seconds as you can".

`int()` guarantees that. It's also why `aurora.audio.sine` uses `int()`.

### Checking the edge case

In the edge tests below you'll see a check for `duration=0.5, sr=3`, there precisely to make sure your implementation follows this convention.

In [ ]:
# ⚡ Shape + value check (gives a hint rather than crashing if not yet implemented)
try:
    freq, dur, sr = 440.0, 1.0, 16000
    mine = my_sine(freq, dur, sr)
    ref = reference_sine(freq, duration=dur, sample_rate=sr)
    max_diff = float(np.max(np.abs(mine - ref)))
    print('my shape:', mine.shape, '| reference shape:', ref.shape)
    print('max pointwise error:', f'{max_diff:.2e}')
    assert mine.shape == ref.shape, 'lengths should match'
    assert max_diff < 1e-12, f'values should be almost identical (machine-precision level); current error={max_diff:.2e}'
    print('\n✅ Answer check passed: your sine wave matches the repo.')
except (NotImplementedError, TypeError):
    print('⚠️  my_sine is not implemented yet; finish the TODO above this cell first, then run this check.')
except TypeError as e:
    print(f'❌ Type error (my_sine may have returned None): {e}')


In [ ]:
# ⚡ Edge tests: covering int() truncation and amplitude=0
try:
    # Edge 1: amplitude=0 should return an all-zero array
    result_zero = my_sine(freq=440.0, duration=1.0, sample_rate=16000, amplitude=0.0)
    assert np.all(result_zero == 0.0), '❌ with amplitude=0 it should return an all-zero array'
    print('✅ Edge 1 passed: amplitude=0 returns an all-zero array')

    # Edge 2: int() truncation (duration=0.5, sr=3 → int(1.5)=1, round(1.5)=2)
    result_trunc = my_sine(freq=1.0, duration=0.5, sample_rate=3)
    ref_trunc = reference_sine(freq=1.0, duration=0.5, sample_rate=3)
    assert result_trunc.shape == ref_trunc.shape, (
        f'❌ int() truncation test: shapes differ, my={result_trunc.shape} vs ref={ref_trunc.shape}\n'
        '   Hint: make sure you use int(duration * sample_rate), not round()'
    )
    print(f'✅ Edge 2 passed: int truncation duration=0.5,sr=3 → N={result_trunc.shape[0]} (matches the repo)')
except (NotImplementedError, TypeError):
    print('⚠️  my_sine is not implemented yet; run the edge tests once it is.')


## 5. Plot it and take a look (the first 50 samples)

A plot shows directly how the sequence changes over time. What you see isn't "an array", but "how this wave swings up and down in time".

The first 50 samples are only a small window, but they help you see:
- whether the waveform is continuous
- whether the amplitude is right
- whether the frequency is what you expect

If you like, you can also overlay `mine[:50]` and `ref[:50]`, which makes any discrepancy easier to spot.


In [ ]:
import matplotlib.pyplot as plt
try:
    mine
except NameError:
    print('⚠️  my_sine is not implemented yet; skipping the plot. Finish the TODO above, then re-run this cell.')
else:
    plt.figure(figsize=(8, 3))
    plt.plot(mine[:50], marker='o', ms=3)
    plt.title('440 Hz sine — first 50 samples')
    plt.xlabel('sample n'); plt.ylabel('amplitude')
    plt.tight_layout(); plt.show()

**🎉 When you're done**: save the notebook (`Ctrl+S`), then commit and push this lesson's changes following the repo conventions.

## 🎨 Picture it: the first 50 samples of a 440 Hz sine

This part isn't "decoration"; it's where a notebook is at its strongest: **turn the formula into a picture, then turn the picture into intuition**. Whenever you write other waveforms, other sequences or other signal-processing code later on, this habit of "generate, compare, visualize" will keep paying off.


## How can you tell the plot is "right"? — compare with wrong implementations

Once you've drawn the waveform, how do you confirm it isn't just similar by coincidence, but actually correct? The best way is to **see what some common mistakes look like**.

### Common mistakes and the plots they produce

**Mistake 1: leaving out 2π, writing `angle = f * n / sr`**
```
x = np.sin(2 * np.pi * freq * n / sr)  ← correct
x = np.sin(freq * n / sr)              ← wrong: 2π left out
```
Effect: the amplitude is unchanged, but the angle grows 2π times too slowly, so the frequency comes out about 6.28 times too low. With `freq=440` you get a sine of roughly 70 Hz — in the first 50 samples it looks like a slow drift instead of more than a full cycle.

**Mistake 2: flipping it, writing `angle = 2π * freq * sr / n`**
```
x = np.sin(2 * np.pi * freq * n / sr)  ← correct
x = np.sin(2 * np.pi * freq * sr / n)  ← wrong: numerator and denominator swapped (and n=0 divides by zero)
```
Effect: with a NumPy array, dividing by zero at n=0 isn't a ZeroDivisionError but a RuntimeWarning and an `inf`, and `sin(inf)` is `nan`; after that you get completely unrecognizable noise.

**Mistake 3: using `round()` instead of `int()`**
```
N = int(duration * sample_rate)    ← correct
N = round(duration * sample_rate)  ← wrong (very close, but edge cases expose it)
```
Effect: most of the time it looks the same, but in edge cases like `duration=0.5, sr=3` the length is off by one, and the shape check in the edge tests catches it.

**Mistake 4: amplitude in the wrong place, e.g. `amplitude * angle` instead of `amplitude * sin(...)`**
```
x = amplitude * np.sin(angle)  ← correct
x = np.sin(amplitude * angle)  ← wrong: it changes the angle, not just the size
```
Effect: the waveform gets "squashed" or "stretched" in time, and the frequency looks completely wrong.

### Three levels of checking correctness

1. **Overall shape**: does it look like a continuously swinging wave? Or noise, a flat line, or wild oscillation?
2. **Period and frequency**: within the first 50 points, do you see the expected number of "rise-and-fall" cycles?
3. **Amplitude range**: is the maximum close to amplitude, and the minimum close to -amplitude?

If all three are right, your implementation is probably correct. If any one of them is off, recheck the formula above.

In [ ]:
from aurora.audio import sine
import aviz as aviz; aviz.style()
aviz.waveform(sine(440.0, duration=50/16000, sample_rate=16000), stem=True,
              title='440 Hz sine — first 50 samples');

In [ ]:
sr = 32
duration = 1.0
t = np.arange(int(duration * sr)) / sr

for freq in [1, 2, 4, 8]:
    x = np.sin(2*np.pi*freq*t)
    zero_crossings = np.sum(np.diff(np.signbit(x)) != 0)
    print(f'freq={freq:>2}Hz | first 8 points={np.round(x[:8], 2)} | zero crossings≈{zero_crossings}')


## An intuitive look at the Nyquist theorem

You may have heard of the "Nyquist frequency", "Nyquist frequency = sr/2", somewhere in your studies — but what does it actually mean?

### The problem: how do you recognize a waveform?

Imagine looking at the sample sequence `[1, 0, -1, 0, 1, 0, -1, 0, ...]` — nothing but these discrete numbers. Can you tell from them "what frequency of sine wave this is"?

The answer: **you need samples that are close enough together.**

If too much time passes between two adjacent samples, a whole "rise and fall" might happen in between, and your sampler misses it completely.

### The limit: what's the fewest points you need?

For a sine wave, at least how many points does it take to describe a full "rise-and-fall" cycle?

Picture the worst case: you sample at `angle=0` (`sin=0`), then sample again right at `angle=π` (`sin=0`). Both points are 0, which looks like "no wave at all". But in between it actually turned 180°, and you completely missed the peak at `angle=π/2`.

To be sure of **not missing the peaks**, you need at least:
- one point in the "rising part" of the cycle (e.g. `sin > 0`)
- one point in the "falling part" of the cycle (e.g. `sin < 0`)

Only then can you tell from the samples changing sign that "there's a wave here".

### The maths: the Nyquist frequency

At sample rate `sr`, the highest frequency that can be represented accurately is `sr/2`.

Why? Because every full cycle needs at least 2 samples:
$$\text{points per cycle} = \frac{sr}{f} \geq 2$$
$$f \leq \frac{sr}{2}$$

When `f = sr/2`, there are exactly 2 points per cycle, e.g. `[1, -1, 1, -1, ...]`. That's the limit — there's no combination of two samples left that can represent anything higher.

### What happens above Nyquist?

If `f > sr/2`, your sampler "folds" the high-frequency signal into a low-frequency one. This is called **aliasing**, and it's the topic of L34.

For example, `f=16 Hz, sr=32 Hz` (Nyquist=16 Hz) is right on the boundary. Go a little higher, to `f=17 Hz`, and it looks like `15 Hz` — that's the "folding".

## Parameter experiment: the sampling limit at the Nyquist frequency

Change `freq` to `sample_rate/2` (the Nyquist frequency) and look at the output: with sin starting at phase 0, every sample lands on a zero crossing, so you get `[0, 0, 0, ...]` and `max=0.0` (with a cosine phase you'd get `[1, -1, 1, -1, ...]`) — only two samples are left per full cycle, which is right at the limit: whether you see the wave at all depends on the phase. This is a preview of L34 (aliasing): when `freq > sample_rate/2` there aren't enough samples, and the high-frequency component gets folded down to a lower frequency.

In [ ]:
sr = 32
duration = 1.0
t = np.arange(int(duration * sr)) / sr

for freq in [1, 2, 4, 8, 16]:  # 16 = sr/2 = the Nyquist frequency
    x = np.sin(2*np.pi*freq*t)
    print(f'freq={freq:>2}Hz | first 8 points={np.round(x[:8], 2)} | max={x.max():.1f}')


## Wrap-up

You can now use `my_sine(freq, duration, sample_rate)` to generate a sine wave of any frequency.

`np.allclose(my_sine(...), reference_sine(...))` returns `True`, which shows the two implementations use the same formula.

`aurora.audio.sine` generates the standard signals in the STFT and mel feature-extraction tests; `my_sine` is your hand-made reproduction of it.

Next lesson (L34) lowers the sample rate on this sine wave and watches how frequencies above Nyquist get folded down to low frequencies.

In [ ]:
# Quick check: start from a short sequence and confirm the output of every step
import numpy as np

sample_rate = 8
duration = 1.0
freq = 1.0
N = round(duration * sample_rate)
n = np.arange(N)
t = n / sample_rate
angle = 2 * np.pi * freq * t
x = np.sin(angle)

print('1) What should N be?', N)
print('2) n is the sample index:', n)
print('3) t is the time of each sample in seconds:', np.round(t, 3))
print('4) angle is the angle of each sample on the circle:', np.round(angle, 3))
print('5) x is the final waveform:', np.round(x, 3))


## Sample count and time alignment: why the "drift" at the end isn't a problem

A careful reader might ask: with `sr=16000, duration=1.0` there are `N=16000` points, so the last point is at `(N-1)/sr = 15999/16000 ≈ 0.9999375 seconds`, `0.0000625 seconds` short. Could this "drift" build up into a problem?

### Why it isn't a problem

In audio processing, a drift on the order of `1/sr` is **an acceptable rounding error**, for three reasons:

1. **Sampling is discretization to begin with**: the whole point of audio sampling is to approximate a continuous signal by its values at a finite set of instants. The samples sit on the grid `n/sr` with `n < N`, so there's no sample at `duration` itself — the next one would belong to the following clip.

2. **The precision is far beyond what's needed**: a drift of `1/16000 ≈ 62.5 microseconds` is completely imperceptible for audio the ear can perceive (on the scale of milliseconds). Your own speakers or microphone already have errors bigger than that.

3. **Joining and processing have slack built in**: in practice, audio segments usually use overlapping windows (as in the STFT), precisely to deal with edge effects. The error at the tail of a single segment gets "blended in".

### When do you need to care?

Only in extreme cases:
- you're doing **ultra-precise signal alignment** (e.g. multitrack mixing that demands nanosecond precision)
- you're doing **long-running accumulation** (e.g. processing 100+ hours of continuous audio, where each segment's small drift adds up)

These are beyond the scope of an introductory audio DSP course. For now, go ahead and use `int()` with confidence.

---
⬇️ **Pass check**: once you've read the wrap-up, finish the whiteboard challenge below before ticking the self-review.


## Reference table for working by hand: key values of sin

To work out the answers to the whiteboard questions you need the sin values of these special angles. No need to memorize them; just remember "quarters of a circle":

| Degrees | Radians | sin value |
|---|---|---|
| 0° | 0 | 0 |
| 90° | π/2 | 1 (peak) |
| 180° | π | 0 (zero crossing) |
| 270° | 3π/2 | -1 (trough) |
| 360° | 2π | 0 (back to the start) |

In whiteboard Question 1, with `freq=1, sr=8`, the angle step between samples is `2π/8 = π/4` (45°). So `n=0,1,2,...,7` correspond to the angles `0, π/4, π/2, 3π/4, π, 5π/4, 3π/2, 7π/4` in turn.

The key point is that **the last point, n=7**, has angle `7π/4`, which hasn't got back to `2π` yet, so `sin(7π/4) ≠ 0`. That's perfectly natural.

## ✏️ Whiteboard challenge: the sine wave by hand (target: 10 minutes)

Cover the screen and answer on paper:

**Question 1**: `freq=1 Hz, sample_rate=8 Hz, amplitude=1, duration=1s`
- N = ? (total number of points)
- x[0] = sin(?) = ?
- x[2] = sin(?) = ? (should be the peak)
- x[4] = sin(?) = ? (back to zero?)

**Question 2**: `freq=440 Hz, sample_rate=16000 Hz, duration=1s`
- N = ?
- x[0] = ?
- How many samples does one cycle take? (`fₛ/f = ?`)

**Question 3**: with amplitude `A=0.5`, the waveform's values lie in `[?, ?]`

**Question 4**: why is `x[-1] ≠ 0`?
- Hint: the last sample is at n=N−1, and its angle is `2π·freq·(N-1)/sr`
- With freq=440, sr=16000, N=16000, that angle equals `2π·440 - 2π·440/16000`
- In other words, it's `2π·440/16000 ≈ 0.172 radians` short of `2π·440` (the end of a whole number of cycles)
- So `sin(that angle) ≠ 0`

When you've finished deriving, run the cell below to check your answers.

In [ ]:
# ✏️ Answer-check cell
import numpy as np

# Question 1: freq=1, sr=8, dur=1
freq, sr, dur = 1.0, 8, 1.0
N1 = round(dur * sr)
assert N1 == 8
n1 = np.arange(N1)
x1 = np.sin(2 * np.pi * freq * n1 / sr)
assert np.isclose(x1[0], 0.0, atol=1e-15)
assert np.isclose(x1[2], 1.0, atol=1e-15), f"x[2] should be 1 (the peak), got {x1[2]:.6f}"
assert np.isclose(x1[4], 0.0, atol=1e-14), f"x[4] should be 0 (a zero crossing), got {x1[4]:.2e}"
print(f"Q1 ✅  N={N1}, x={np.round(x1, 4)}")
print(f"      x[2]={x1[2]:.4f} (peak), x[4]={x1[4]:.2e} (zero crossing)")

# Check that my_sine agrees
try:
    mine = my_sine(freq, dur, sr)
    assert np.allclose(mine, x1, atol=1e-12)
    print(f"      my_sine matches the hand calculation ✅")
except (NotImplementedError, TypeError):
    print("⬜ Q1: implement my_sine() first, then run the answer-check cell")

# Question 2: 440 Hz, sr=16000
N2 = round(1.0 * 16000)
assert N2 == 16000
samples_per_cycle = 16000 / 440
assert np.isclose(samples_per_cycle, 36.36, atol=0.01)
print(f"Q2 ✅  N=16000, fₛ/f=16000/440≈{samples_per_cycle:.2f} samples per cycle")

# Question 3: amplitude=0.5
x3 = 0.5 * np.sin(2 * np.pi * 1.0 * np.arange(8) / 8)
assert np.max(x3) <= 0.5 + 1e-12 and np.min(x3) >= -0.5 - 1e-12
print(f"Q3 ✅  amplitude=0.5, value range=[{np.min(x3):.4f}, {np.max(x3):.4f}]")

# Question 4: the last sample x[-1] doesn't land back on zero
x4 = np.sin(2 * np.pi * 440 * np.arange(16000) / 16000)
last = x4[-1]
print(f"Q4 ✅  440Hz/16000Hz → x[-1]={last:.6f}≠0 (the last point is at n=N-1, one sample interval before the end of a whole number of cycles)")
print(f"      the angle of x[-1] = 2π·440·15999/16000 = 2π·440 − 2π·440/16000, not an integer multiple of 2π")
print("\n🎉 Sine-wave whiteboard challenge passed! x[n]=A·sin(2πf·n/fₛ) is now second nature.")

In [ ]:
# ✏️ Self-review for this lesson
l33_review = {
    "sine_formula":           None,  # remember x[n]=A·sin(2πf·n/fₛ)? True/False
    "my_sine_implemented":    None,  # implemented my_sine and passed the allclose assertion? True/False
    "samples_per_cycle":      None,  # understand samples per cycle = fₛ/f? True/False
    "nyquist_intuition":      None,  # know that sampling only works properly when f < fₛ/2 (Nyquist)? True/False
    "whiteboard_passed":      None,  # finished the whiteboard challenge on paper? True/False
}

unfilled = [k for k, v in l33_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l33_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L33 fully cleared! On to L34: Nyquist and aliasing')

---

→ **Next lesson** · [L34 · Nyquist and aliasing](L34_aliasing.ipynb)

> Next lesson covers **Nyquist and aliasing**: what a 6 kHz sine wave becomes when it's sampled at 8 kHz.